# Filter Denver's Indels

This notebook filters the annotated-varscan indel table by VAF and depth across two
replicates, plus a gnomAD population-frequency cutoff.

> **Disclaimer:** This notebook filters Denver's indels down to what *appears to be real
> indels*, based on VAF, depth, and support by two replicates. **It does not apply any
> further filtering for haematopoietic-relevant genes or for exonic/functional consequence**
> — intronic indels are kept. If that kind of filtering is required, apply it yourself on
> the output file.

It is configured to filter Denver's annotated varscan table
(**`data/annotated_varscan_filtered_by_gene_function_orig.csv`**). To run it, **Run All**.
One TSV file is written to the `out/` folder.

Everything is parameterised, so you can point it at a different file by editing one cell.

## 1. Imports

In [19]:
import os
import re
import numpy as np
import pandas as pd

## 2. Parameters
Column names differ from the SNV table; edit thresholds / names here.

In [20]:
P = dict(
    # --- I/O ---
    input_file="data/annotated_varscan_filtered_by_gene_function_NovaseqR99.csv",
    output_dir="out",
    run_name="annotated_varscan",

    # --- column names as they appear in the INPUT table (the coordinate ones are renamed
    #     to CHR/POS/REF/ALT right after reading, see section 3) ---
    gene_column="Gene.refGene",
    chr_column="Chr",
    pos_column="Start",
    ref_column="Ref",
    alt_column="Alt",
    gnomad_column="gnomad40_genome_AF",
    depth_columns=["Depth_dup1", "Depth_dup2"],
    vaf_columns=["VAF", "VAF_DUP"],

    # --- filters ---
    min_VAF_both=0.02,      # require VAF AND VAF_DUP strictly greater than this
    min_depth=20,           # require each duplicate depth >= this
    max_gnomad_AF=0.001,
)

os.makedirs(P["output_dir"], exist_ok=True)
def log(msg): print("[LOG]", msg)

## 3. Read input & validate
The coordinate columns are renamed to `CHR` / `POS` / `REF` / `ALT` right after reading, so the
output matches the SNV notebook's naming.

In [21]:
df = pd.read_csv(P["input_file"], low_memory=False)
log(f"Input file {P['input_file']} read: {df.shape[0]} rows, {df.shape[1]} columns")

required = ([P["gene_column"], P["chr_column"], P["pos_column"], P["ref_column"],
            P["alt_column"], P["gnomad_column"], "Func.refGene", "ExonicFunc.refGene",
            "AAChange.refGene"] + P["vaf_columns"] + P["depth_columns"])
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
assert len(df) > 0, "No rows in input!"

# Standardise the coordinate columns to CHR/POS/REF/ALT (the names the SNV notebook's output
# uses), so downstream consumers can read the SNV and indel outputs the same way.
rename_map = {P["chr_column"]: "CHR", P["pos_column"]: "POS",
              P["ref_column"]: "REF", P["alt_column"]: "ALT"}
df = df.rename(columns=rename_map)
log(f"Renamed coordinate columns: {rename_map}")

df_f = df.copy()

[LOG] Input file data/annotated_varscan_filtered_by_gene_function_NovaseqR99.csv read: 4253223 rows, 18 columns
[LOG] Renamed coordinate columns: {'Chr': 'CHR', 'Start': 'POS', 'Ref': 'REF', 'Alt': 'ALT'}


## 4. Keep indels only
Annovar-style indels: `Ref` or `Alt` is `-`, or `Ref`/`Alt` differ in length.

In [22]:
ref = df_f["REF"].astype(str)
alt = df_f["ALT"].astype(str)
is_indel = (ref == "-") | (alt == "-") | (ref.str.len() != alt.str.len())
df_f = df_f[is_indel]
log(f"After keeping indels only: {len(df_f)} rows")
assert len(df_f) > 0, "No indels found!"

[LOG] After keeping indels only: 397 rows


## 5. Filter by VAF — both duplicates > P["min_VAF_both"]

In [23]:
for c in P["vaf_columns"]:
    if not pd.api.types.is_numeric_dtype(df_f[c]):
        s = df_f[c].astype(str)
        if s.str.endswith("%").all():
            df_f[c] = s.str.replace("%", "", regex=False).astype(float) / 100
            log(f"Converted {c} percentage strings to numeric.")
        else:
            df_f[c] = pd.to_numeric(df_f[c], errors="coerce")

f_vaf = pd.Series(True, index=df_f.index)
for c in P["vaf_columns"]:
    f_vaf &= df_f[c] > P["min_VAF_both"]
df_f = df_f[f_vaf]
log(f"After {' & '.join(P['vaf_columns'])} > {P['min_VAF_both']}: {len(df_f)} rows")
assert len(df_f) > 0, "No indels left after VAF filtering!"

[LOG] After VAF & VAF_DUP > 0.02: 204 rows


## 6. Filter by depth

In [24]:
for c in P["depth_columns"]:
    df_f = df_f[pd.to_numeric(df_f[c], errors="coerce") >= P["min_depth"]]
    log(f"After {c} >= {P['min_depth']}: {len(df_f)} rows")
assert len(df_f) > 0, "No indels left after depth filtering!"

[LOG] After Depth_dup1 >= 20: 194 rows
[LOG] After Depth_dup2 >= 20: 179 rows


## 7. Average VAF & gnomAD AF filter
Computes `avg_VAF` across the two duplicates, then filters by gnomAD population AF.

In [25]:
df_f["avg_VAF"] = df_f[P["vaf_columns"]].apply(pd.to_numeric, errors="coerce").mean(axis=1)

df_f["numeric_gnomad_AF"] = pd.to_numeric(df_f[P["gnomad_column"]], errors="coerce")
df_f = df_f[df_f["numeric_gnomad_AF"].isna() |
            (df_f["numeric_gnomad_AF"] <= P["max_gnomad_AF"])]
log(f"After gnomAD AF <= {P['max_gnomad_AF']}: {len(df_f)} rows")
assert len(df_f) > 0, "No indels left after gnomAD filtering!"

[LOG] After gnomAD AF <= 0.001: 173 rows


## 8. Add variant_count & write output file

In [26]:
variant_keys = ["CHR", "POS", "REF", "ALT"]
df_f["variant_count"] = df_f.groupby(variant_keys)["CHR"].transform("size")

output_fn = os.path.join(P["output_dir"], f"filtering_Indels_{P['run_name']}_filtered.tsv")
df_f.to_csv(output_fn, index=False, na_rep="NA", sep="\t")
log(f"Wrote {output_fn}  ({len(df_f)} rows)")

[LOG] Wrote out/filtering_Indels_annotated_varscan_filtered.tsv  (173 rows)


## 9. Preview

In [27]:
cols = ["Gene.refGene", "Func.refGene", "ExonicFunc.refGene", "avg_VAF", "variant_count"]
cols = [c for c in cols if c in df_f.columns]
print("Indel genes:", df_f["Gene.refGene"].value_counts().to_dict())
df_f[cols].head(20)

Indel genes: {'DNMT3A': 43, 'TET2': 42, 'ASXL1': 36, 'CEBPA': 24, 'PPM1D': 13, 'ZRSR2': 4, 'PDS5B': 3, 'BRCC3': 3, 'NF1': 2, 'RUNX1': 1, 'BCORL1': 1, 'EZH2': 1}


,Gene.refGene,Func.refGene,ExonicFunc.refGene,avg_VAF,variant_count
15975,DNMT3A,exonic,frameshift deletion,0.03325,1
23028,RUNX1,exonic,nonframeshift deletion,0.31595,1
60616,CEBPA,exonic,nonframeshift deletion,0.02445,24
108039,DNMT3A,exonic,frameshift deletion,0.07240,1
133240,TET2,exonic,frameshift insertion,0.04805,1
159593,CEBPA,exonic,nonframeshift deletion,0.03310,24
181919,DNMT3A,exonic,frameshift deletion,0.09010,1
195655,NF1,exonic,frameshift deletion,0.05795,1
207030,DNMT3A,exonic,frameshift deletion,0.02675,1
215379,CEBPA,exonic,nonframeshift deletion,0.06120,24
